# The extent-of-resection "double-duty" problem — an interactive explainer

This notebook lets you *see*, with the sliders below, why a standard survival
model can report the **same biology as opposite conclusions** depending on the
mix of tumors in a cohort.

**What this is:** a teaching simulation. We build synthetic patients whose true
causal structure we *set ourselves*, then fit the model everyone fits.

**What this is not:** a validated clinical model. Nothing here should inform a
decision about a real patient.

---
### The structure we impose

```
                    ┌── (direct, protective) ─────────────────┐
extent of resection ┤                                          ├─→ hazard
                    └── new deficit ─→ ↓KPS ─→ (harmful) ──────┘
                              ▲
                     steeper in eloquent tumors   ← the mechanism
```

Extent of resection helps by removing tumor and hurts by causing deficits. The
deficit path is steeper in eloquent tumors. Every other parameter is held fixed
across cohorts — only the **eloquent fraction** changes.

In [ ]:
# If running on Colab, uncomment:
# !pip install eor-sim ipywidgets matplotlib
import numpy as np, matplotlib.pyplot as plt
from eor_sim import SimParams, sweep_eloquent_fraction
from eor_sim.plotting import plot_sweep

### 1. The headline result

The dashed line is the **true** direct effect of resection. It is identical in
every cohort. Watch the fitted points (the conventional model) drift away from
it as the eloquent fraction rises.

In [ ]:
sweep = sweep_eloquent_fraction(estimator="total")
ax = plot_sweep(sweep, SimParams(),
                title="Same biology, opposite conclusions across cohorts")
plt.show()
print(sweep.to_string(index=False))

### 2. Turn the mechanism up and down (interactive)

`d_eor_x_eloq` is the extra steepness of the deficit path in eloquent tumors —
**the mechanism**. Set it to **0** and the drift should vanish: that is the
built-in falsification check. If the slope did *not* vanish at 0, the estimator
would be detecting something we never put in, and nothing downstream could be
trusted.

In [ ]:
from ipywidgets import interact, FloatSlider

def show(d_eor_x_eloq=3.0, deficit_kps_drop=25.0, b_eor_direct=-1.30):
    p = SimParams(d_eor_x_eloq=d_eor_x_eloq,
                  deficit_kps_drop=deficit_kps_drop,
                  b_eor_direct=b_eor_direct)
    sweep = sweep_eloquent_fraction(estimator="total", params=p, reps=20)
    ax = plot_sweep(sweep, p)
    slope = np.polyfit(sweep.eloquent_fraction, sweep.coef_mean, 1)[0]
    ax.set_title(f"slope = {slope:+.3f}   (→ 0 means no artifact)",
                 fontsize=10, fontweight="bold")
    plt.show()

interact(show,
         d_eor_x_eloq=FloatSlider(min=0, max=5, step=0.5, value=3.0),
         deficit_kps_drop=FloatSlider(min=0, max=40, step=5, value=25.0),
         b_eor_direct=FloatSlider(min=-2.0, max=0.0, step=0.1, value=-1.30));

### 3. The inversion: the "cleaner" model is the wrong one

Conditioning on **post-operative** KPS recovers the true direct effect and is
stable across cohorts. It is also the wrong model for a surgical decision,
because it holds constant a variable (post-op function) that the surgery itself
caused — deleting the cost of the operation from the estimate of its benefit.

In [ ]:
total  = sweep_eloquent_fraction(estimator="total")
direct = sweep_eloquent_fraction(estimator="direct")
print("TOTAL effect (conventional; what a surgeon needs):")
print(total[["eloquent_fraction","coef_mean"]].to_string(index=False))
print("\nDIRECT effect (conditions on post-op KPS; recovers truth, wrong for the decision):")
print(direct[["eloquent_fraction","coef_mean"]].to_string(index=False))

### Where to go next

- **Reproduce the paper figure:** `python scripts/reproduce_figure.py`
- **Run on your own cohort:** see `examples/run_on_your_data.py`
- **Read the caveats:** the README section *"What this is and is not"* — in
  particular, post-operative KPS is really a *convergence point* (baseline KPS,
  age, and deficit all feed it), which makes the honest analysis a
  causal-mediation problem, not a regression with more terms.